In [7]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, 
    roc_auc_score, 
    precision_score, 
    recall_score, 
    f1_score, 
    matthews_corrcoef
)

# --- 4. LOAD AND SPLIT DATASET ---
df = pd.read_csv('bank.csv', sep=';')

X = df.drop(columns=['y'])
y = df['y'].apply(lambda x: 1 if x == 'yes' else 0)

# FIX: Explicitly include both 'object' and 'str' to handle Pandas 3/4 string migration
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object', 'str']).columns.tolist()

# --- 5. PREPROCESSING PIPELINE ---
numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# --- 6. DEFINE YOUR 5 SPECIFIED MODELS ---
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=6, random_state=42),
    "K-Nearest Neighbor": KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
    "Gaussian Naive Bayes": GaussianNB(),
    "Support Vector Machine": CalibratedClassifierCV(LinearSVC(dual=False, random_state=42))
}


In [8]:
# --- 5. TRAIN AND EVALUATE METRICS ---
results = {}

In [9]:
print("\n--- Training Models and Calculating All Metrics ---")
for name, model in models.items():
    pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                               ('model', model)])
    
    pipeline.fit(X_train, y_train)
    
    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1] 
    
    # Calculate all 6 required evaluation metrics
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)
    
    results[name] = {
        "Accuracy": acc,
        "AUC Score": auc,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "MCC Score": mcc
    }
    
    print(f"\n================ {name} ================")
    print(f"1. Accuracy:                 {acc:.4f}")
    print(f"2. AUC Score:                {auc:.4f}")
    print(f"3. Precision:                {precision:.4f}")
    print(f"4. Recall:                   {recall:.4f}")
    print(f"5. F1 Score:                 {f1:.4f}")
    print(f"6. MCC Score:                {mcc:.4f}")

# --- 6. SUMMARY COMPARISON TABLE ---
print("\n================ FINAL ARCHITECTURE PERFORMANCE LEADERBOARD ================")
summary_df = pd.DataFrame(results).T
# Sorted by MCC because it handles imbalanced data distributions robustly
print(summary_df.sort_values(by="MCC Score", ascending=False).round(4))


--- Training Models and Calculating All Metrics ---

================ Logistic Regression ================
1. Accuracy:                 0.9166
2. AUC Score:                0.9424
3. Precision:                0.7118
4. Recall:                   0.4364
5. F1 Score:                 0.5411
6. MCC Score:                0.5162

================ Decision Tree ================
1. Accuracy:                 0.9199
2. AUC Score:                0.9346
3. Precision:                0.6763
4. Recall:                   0.5539
5. F1 Score:                 0.6090
6. MCC Score:                0.5684

================ K-Nearest Neighbor ================
1. Accuracy:                 0.9075
2. AUC Score:                0.8799
3. Precision:                0.6250
4. Recall:                   0.4472
5. F1 Score:                 0.5214
6. MCC Score:                0.4798

================ Gaussian Naive Bayes ================
1. Accuracy:                 0.8203
2. AUC Score:                0.8393
3. Precision: